In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (average_precision_score, precision_recall_curve,
                             precision_score, recall_score, f1_score, cohen_kappa_score)
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit

ROOT = Path.cwd().parent
df = pd.read_csv(ROOT / "annonated_training_test_set.csv").dropna(subset=["review"]).reset_index(drop=True)
risk_labels = ["label_dependence", "label_attachment",
               "label_isolation_displacement", "label_crisis_response_frustration"]
short = [l.replace("label_", "") for l in risk_labels]
Y_all = (df[risk_labels] == "Yes").astype(int)

# Save the test IDs once and commit the file. Every later run reloads the same 60 reviews.
TEST_PATH = ROOT / "data" / "heldout_test_uids.csv"
if TEST_PATH.exists():
    test_uids = set(pd.read_csv(TEST_PATH)["UID"])
else:
    sss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    _, te_idx = next(sss.split(df["review"], Y_all))
    test_uids = set(df.loc[te_idx, "UID"])
    pd.DataFrame({"UID": sorted(test_uids)}).to_csv(TEST_PATH, index=False)

is_test = df["UID"].isin(test_uids)
dev = df[~is_test].reset_index(drop=True)     # all development work uses this
test = df[is_test].reset_index(drop=True)     # do not touch until the final evaluation
Y_dev = (dev[risk_labels] == "Yes").astype(int)
Y_test = (test[risk_labels] == "Yes").astype(int)

In [2]:
dist = pd.DataFrame({"dev (train + CV)": Y_dev.sum(), "held-out test": Y_test.sum()})
dist.index = short
display(dist)

folds = list(MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
             .split(dev["review"], Y_dev))
display(pd.DataFrame({f"fold {i}": Y_dev.iloc[te].sum().values
                      for i, (_, te) in enumerate(folds)}, index=short))

,dev (train + CV),held-out test
dependence,11,3
attachment,17,4
isolation_displacement,15,4
crisis_response_frustration,18,5


,fold 0,fold 1,fold 2,fold 3,fold 4
dependence,2,3,2,2,2
attachment,4,3,3,4,3
isolation_displacement,3,3,3,3,3
crisis_response_frustration,4,3,3,4,4


In [3]:
co = Y_all.T @ Y_all
co.index = co.columns = short
co   # diagonal = total per theme; off-diagonal = reviews with both themes

,dependence,attachment,isolation_displacement,crisis_response_frustration
dependence,14,5,3,0
attachment,5,21,4,0
isolation_displacement,3,4,19,0
crisis_response_frustration,0,0,0,23


In [4]:
C_GRID = [0.1, 0.3, 1, 3, 10, 30]

def make_model(C):
    return make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
        LogisticRegression(C=C, class_weight="balanced", max_iter=5000))

def best_f1_threshold(y, p):
    pr, rc, t = precision_recall_curve(y, p)
    f1 = 2 * pr * rc / (pr + rc + 1e-9)
    return t[np.argmax(f1[:-1])]

def tune_and_fit(X_tr, y_tr):
    """Pick C and the threshold with an inner CV, then fit on all of X_tr."""
    inner = StratifiedKFold(3, shuffle=True, random_state=42)
    probs = {C: cross_val_predict(make_model(C), X_tr, y_tr, cv=inner, method="predict_proba")[:, 1]
             for C in C_GRID}
    best_C = max(C_GRID, key=lambda C: average_precision_score(y_tr, probs[C]))
    thr = best_f1_threshold(y_tr, probs[best_C])
    return make_model(best_C).fit(X_tr, y_tr), thr

def run_cv(texts, Y, folds):
    proba = pd.DataFrame(0.0, index=Y.index, columns=Y.columns)
    pred = pd.DataFrame(0, index=Y.index, columns=Y.columns)
    for tr, te in folds:
        for label in Y.columns:
            model, thr = tune_and_fit(texts.iloc[tr], Y[label].iloc[tr])
            p = model.predict_proba(texts.iloc[te])[:, 1]
            proba.iloc[te, proba.columns.get_loc(label)] = p
            pred.iloc[te, pred.columns.get_loc(label)] = (p >= thr).astype(int)
    return proba, pred

def score(Y, proba, pred):
    rows = {}
    for label in Y.columns:
        rows[label.replace("label_", "")] = {
            "precision": precision_score(Y[label], pred[label], zero_division=0),
            "recall": recall_score(Y[label], pred[label], zero_division=0),
            "F1": f1_score(Y[label], pred[label], zero_division=0),
            "AP": average_precision_score(Y[label], proba[label])}
    none_t = (Y.sum(axis=1) == 0).astype(int)
    none_p = (pred.sum(axis=1) == 0).astype(int)
    rows["other_none"] = {"precision": precision_score(none_t, none_p, zero_division=0),
                          "recall": recall_score(none_t, none_p, zero_division=0),
                          "F1": f1_score(none_t, none_p, zero_division=0), "AP": np.nan}
    out = pd.DataFrame(rows).T
    out.loc["MACRO F1 (5 classes)", "F1"] = out["F1"].iloc[:5].mean()
    out.loc["MACRO F1 (4 risk themes)", "F1"] = out["F1"].iloc[:4].mean()
    return out

In [5]:
APP_NAMES = [
    "woebot", "woe bot", "wysa", "youper", "amaha", "innerhour", "inner hour",
    "7 cups", "7cups", "seven cups", "sevencups", "healthily", "your.md", "happify",
    "voidpet", "void pet", "noni", "allie",
]
name_re = re.compile(r"\b(" + "|".join(re.escape(n) for n in APP_NAMES) + r")\b", re.IGNORECASE)
# MHARD already deleted most app names and left a double space; mark those gaps too
gap_re = re.compile(r"(?<=\w) {2,}(?=\w)|(?<=\w) {2,}(?=[.,!?)])|(?<=[(]) {1,}(?=\w)")

def mask_names(text):
    return name_re.sub("appname", text)

def mask_names_and_gaps(text):
    return mask_names(gap_re.sub(" appname ", text))   # gaps first, so masking adds no new gaps

variants = {
    "original": dev["review"],
    "mask names": dev["review"].map(mask_names),
    "mask names + gaps": dev["review"].map(mask_names_and_gaps),
}
print("Reviews changed:", {k: int((v != dev["review"]).sum()) for k, v in variants.items()})

results, oof = {}, {}
for name, texts in variants.items():
    proba, pred = run_cv(texts, Y_dev, folds)
    oof[name] = (proba, pred)
    results[name] = score(Y_dev, proba, pred)

pd.concat({k: v[["precision", "recall", "F1"]] for k, v in results.items()}, axis=1).round(3)

Reviews changed: {'original': 0, 'mask names': 10, 'mask names + gaps': 18}


original               mask names                \
                            precision recall     F1  precision recall     F1   
dependence                      0.040  0.455  0.074      0.040  0.455  0.074   
attachment                      0.241  0.412  0.304      0.226  0.412  0.292   
isolation_displacement          0.217  0.333  0.263      0.217  0.333  0.263   
crisis_response_frustration     0.375  0.500  0.429      0.346  0.500  0.409   
other_none                      0.814  0.374  0.513      0.826  0.380  0.520   
MACRO F1 (5 classes)              NaN    NaN  0.316        NaN    NaN  0.312   
MACRO F1 (4 risk themes)          NaN    NaN  0.267        NaN    NaN  0.259   

                            mask names + gaps                
                                    precision recall     F1  
dependence                              0.040  0.455  0.074  
attachment                              0.250  0.412  0.311  
isolation_displacement                  0.190  0.267  0.222  
crisis_response_frustration             0.400  0.556  0.465  
other_none                              0.835  0.406  0.547  
MACRO F1 (5 classes)                      NaN    NaN  0.324  
MACRO F1 (4 risk themes)                  NaN    NaN  0.268

In [6]:
app_ap = {}
for label in risk_labels:
    p = pd.Series(0.0, index=dev.index)
    for tr, te in folds:
        m = make_pipeline(OneHotEncoder(handle_unknown="ignore"),
                          LogisticRegression(class_weight="balanced", max_iter=2000))
        m.fit(dev[["app_name"]].iloc[tr], Y_dev[label].iloc[tr])
        p.iloc[te] = m.predict_proba(dev[["app_name"]].iloc[te])[:, 1]
    app_ap[label.replace("label_", "")] = average_precision_score(Y_dev[label], p)

pd.DataFrame({"base rate": Y_dev.mean().values,
              "AP, app name only": list(app_ap.values()),
              "AP, review text": results["original"]["AP"].iloc[:4].values}, index=short).round(3)

,base rate,"AP, app name only","AP, review text"
dependence,0.046,0.052,0.060
attachment,0.071,0.093,0.211
isolation_displacement,0.062,0.051,0.121
crisis_response_frustration,0.075,0.116,0.388


In [7]:
proba, pred = oof["original"]
err = dev[["UID", "app_name", "rating", "review"]].copy()
for l, s in zip(risk_labels, short):
    err[f"true_{s}"] = Y_dev[l]
    err[f"pred_{s}"] = pred[l]
err["proba_dependence"] = proba["label_dependence"].round(3)
err["error_type"] = np.select(
    [(err.true_dependence == 1) & (err.pred_dependence == 0),
     (err.true_dependence == 0) & (err.pred_dependence == 1)],
    ["missed (FN)", "false alarm (FP)"], default="")
err = err[err.error_type != ""].sort_values(["error_type", "proba_dependence"], ascending=[True, False])
err["reviewer_1"] = ""
err["reviewer_2"] = ""
err["notes"] = ""
err.to_csv(ROOT / "data" / "dependence_errors_to_review.csv", index=False)
err["error_type"].value_counts()

error_type
false alarm (FP)    120
missed (FN)           6
Name: count, dtype: int64